# Wolf-of-Wallstreet — train on Google Colab (clean v2)

**Every problem we hit is fixed in `scripts/pretrain.py`:** the Binance geo-block (HTTP 451), the 2025 µs-timestamp crash, the RAM out-of-memory (“^C”) during sequence building, the PyTorch `verbose` crash, **and the disk overflow** (no more 41 GB second copy). Plus two model upgrades: **recency-weighting** (the model leans toward the current market regime) and **AMP mixed-precision** (2–3× faster on L4/A100).

### 3 steps
1. **Runtime → Change runtime type → L4 GPU** (best value with compute units; pick **A100** for the fastest finish). Save.
2. Put your **`trading-agent`** folder in Google Drive (*My Drive*) with `scripts/`, `backend/`, `requirements.txt`, **and your `.env`** (it holds your Alpaca keys + the tuned knobs). You can delete `frontend/`, `backend/.venv/`, `.git/`, `node_modules/`, and `training_data/` to keep the upload small — data re-downloads and caches automatically.
3. **Runtime → Run all.** Approve the Drive popup on the first cell.

The checkpoint is written to **`models/trading_lstm_latest.pt` in your Drive** (survives disconnects). Bring it back to your PC's `trading-agent/models/`.

> **You run directly from the Drive folder** (no copy-to-local), so your `.env` + cached data are found automatically and the checkpoint persists. Only the big float16 dataset is written to fast **local** disk (`/content/_mmap_cache`), which is **wiped at the start of every run** so it can never overflow the 112 GB disk.

> **Keep the tab active** during training — idle Colab tabs get reclaimed. The checkpoint is saved on each new best epoch, so a disconnect after that keeps your progress.

### 1) Mount Drive + verify the project
Edit `PROJECT_DIR` if your folder is named/placed differently. This also checks your uploaded `pretrain.py` actually has all the fixes (so you never run a stale copy again).

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/trading-agent"   # <-- edit if your folder differs

from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR); sys.path.insert(0, os.path.join(PROJECT_DIR, 'backend'))
print('Working dir :', os.getcwd())
print('.env present:', os.path.exists('.env'))

# Confirm the uploaded files have ALL the latest fixes (re-upload whatever this flags).
pt  = open('scripts/pretrain.py', encoding='utf-8').read()
bt  = open('scripts/backtest.py', encoding='utf-8').read()
eng = open('backend/backtest/engine.py', encoding='utf-8').read()
fsp = open('backend/signals/feature_spec.py', encoding='utf-8').read()
need_pt  = ['data-api.binance.vision', 'triple_barrier_labels', 'make_split_loaders',
            'feat_path', 'assemble_feature_matrix',          # shared assembly (backtest width fix)
            'Always prefer the cache',                       # --skip-download cache fix
            'TEST_FRAC',                                     # P1a: reserved hold-out split
            'assemble_with_cache',                           # A3: prebuilt-feature cache
            'rolling_vwap_distance',                         # A2: fixed (alive) VWAP
            '_by_close_time']                                # HTF look-ahead leak fix (audit-found)
need_bt  = ['assemble_feature_matrix', 'primary-horizon', 'out-json', 'FEATURE-WIDTH MISMATCH',
            'excess_sharpe']                                 # alpha-vs-beta metric
need_eng = ['ann_factor', 'excess_sharpe']                  # P0 Sharpe fix + alpha metric
need_fs  = ['rolling_vwap_distance', 'VWAP_WINDOW']
miss = ([n for n in need_pt if n not in pt] + [n for n in need_bt if n not in bt]
        + [n for n in need_eng if n not in eng] + [n for n in need_fs if n not in fsp])
print('script fixes:', '✅ all present' if not miss else f'❌ MISSING {miss} — re-upload the file(s)')

# Feature width + architecture must be IDENTICAL here and wherever the checkpoint loads.
from backend.signals import feature_spec as fs
from backend.core.config import settings as _s
print(f'feature_spec {fs.VERSION}: INPUT={fs.INPUT}  (base {fs.BASE} + htf {fs.HTF} + news {fs.NEWS_EMBED_DIM} + earnings {fs.EARNINGS_DIM})')
print(f'model: hidden={_s.NN_HIDDEN_SIZE}  layers={_s.NN_NUM_LAYERS}  dropout={_s.NN_DROPOUT}  (P1c smaller trunk)')
print('Phase A: causal feats + HTF leak fix + rolling VWAP + BB-width fix + feature cache.')
print('NOTE: architecture changed (256/3 → 128/2). OLD ckpt_*.pt will NOT load — retrain fresh.')

assert os.path.exists('.env'), 'No .env — upload it (Alpaca keys + tuned knobs).'
assert not miss, 'Stale files — re-upload the flagged file(s) to Drive, then re-run.'

### 2) Install dependencies
Tiny + pure-Python. **No `pandas_ta`, no numpy/pandas pins** — `pretrain.py` has a built-in indicator shim, so Colab's default numpy 2 + pandas stay put (no conflicts). Ignore any ‘fix pandas’ suggestions.

In [ ]:
!pip install -q structlog python-dotenv pydantic pydantic-settings requests scikit-learn
!pip install -q sentence-transformers   # semantic news embeddings (NN_NEWS_EMBED_BACKEND=transformer)
!python -c "import numpy,pandas,torch,sklearn; print('OK | numpy',numpy.__version__,'| pandas',pandas.__version__,'| torch',torch.__version__)"

### 3) Confirm the GPU + disk
The dataset build is CPU/disk-bound; the GPU only engages once training starts. A faster GPU (L4/A100) speeds the **epochs**, not the build.

In [ ]:
import torch, shutil
print('CUDA :', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU  :', p.name, f'({p.total_memory/1e9:.0f} GB)')
else:
    print('No GPU — Runtime → Change runtime type → L4 (or T4), then Run all again.')
t, u, f = shutil.disk_usage('/content')
print(f'Disk /content: {f/1e9:.0f} GB free of {t/1e9:.0f} GB  (on-the-fly windows → dataset is only ~1-2 GB)')

### 4) Training knobs (read from your uploaded `.env`)
Already tuned for generalization + risk-adjusted returns, and read from `.env` so they match what you'll run live. **Don't override them here.** `NN_RECENCY_*` is what makes the model emphasise newer data (current regime).

In [ ]:
from dotenv import dotenv_values
cfg = dotenv_values('.env')
print('— model / regularization —')
for k in ['NN_RNN_TYPE','NN_DROPOUT','NN_WEIGHT_DECAY','NN_LABEL_SMOOTHING','NN_AUGMENT_NOISE_STD']:
    print(f'  {k:26s}= {cfg.get(k)}')
print('— recency weighting (newer data emphasised) —')
for k in ['NN_RECENCY_HALFLIFE_YEARS','NN_RECENCY_FLOOR']:
    print(f'  {k:26s}= {cfg.get(k) or "(default 2.0 / 0.25)"}')
print('— reward / sizing (live) —', {k: cfg.get(k) for k in ['NN_KELLY_FRACTION','NN_MIN_EDGE_OVER_FEE','RISK_MAX_POSITION_PCT']})
assert cfg.get('ALPACA_API_KEY'), 'No ALPACA_API_KEY in .env — the 5 stocks would be skipped.'
print('\nALPACA key loaded ✅  — the 5 stocks will train.')

### 5) Three experiments, back-to-back: LSTM → LSTM+news → TCN

Each experiment trains **from scratch**, copies its checkpoint to a **named file in Drive**, then backtests it **out-of-sample on the 1h horizon** (the model's real edge — the 15‑min head was noise) and writes a metrics JSON. The last two cells compare all three and tell you which checkpoint to keep.

- **Run the cells top-to-bottom.** Each experiment cell does **train → save → backtest** in one go.
- **Data is downloaded once.** The prefetch cell below pulls all raw OHLCV into the **Drive parquet cache**; every experiment then runs with `--skip-download`, so nothing re-downloads between cells. (Separate `!python` cells are separate processes, so they can't share RAM — the on-disk cache is the cross-cell store.)
- **Each rebuilds its own feature matrix** (the news run needs real news features; the others use zeros), so the three are comparable. The news run's backtest sets `NEWS_ALIGN=1` so its news columns are populated.
- Rough time on L4: **~40–50 min per experiment** (the news run a bit longer). Keep the tab active.

In [ ]:
# Shared knobs for all three experiments. Model size + regularization still come from your .env.
START_YEAR = 2022     # full history (fast now via on-the-fly windows)
EPOCHS     = 50       # high is fine — early stopping ends it when val stops improving
PATIENCE   = 8
BATCH_SIZE = 512      # 1024 on A100 / 256 on T4
SEED       = 42
PRIMARY_H  = 1        # horizon to BACKTEST: 1 = H+12 (1h). Try 2 = H+48 (4h) — it had the best Sharpe.
SYMBOLS    = ("BTCUSDT ETHUSDT SOLUSDT AAVEUSDT XLMUSDT XRPUSDT ADAUSDT DOGEUSDT RENDERUSDT NEARUSDT "
              "SNDK AMD MU AXTI BE NVDA TSM SMCI")
import os; os.makedirs('models', exist_ok=True)
print(f'config: START_YEAR={START_YEAR}  EPOCHS={EPOCHS}  BATCH={BATCH_SIZE}  backtest horizon index={PRIMARY_H}')

In [ ]:
# Prefetch ALL raw OHLCV into the Drive parquet cache ONCE, so the experiments below read
# from cache (they pass --skip-download) and never re-download per cell.
import importlib.util
_spec = importlib.util.spec_from_file_location('pre', 'scripts/pretrain.py')
pre = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(pre)
for s in SYMBOLS.split():
    try:
        pre.load_full_history(s, START_YEAR, 1, skip_download=False)   # downloads if missing, else loads cache
        print('cached', s)
    except Exception as e:
        print('skip', s, '-', str(e)[:80])
print('\n✅ raw data cached on Drive — experiments use --skip-download (no per-cell re-download)')

In [ ]:
# ===== COMPREHENSIVE SIGNAL AUDIT — crypto + BROAD stock universe (measure-first, no training) =====
# Reports per-horizon baseline AUC (base 90 features  vs  +Phase-B candidates incl BTC cross-asset
# context for alts) + the feature IC ranking. We add a BROAD stock set (our 8 + ~16 related semis)
# to test whether ANY stock has a 5m edge — NVDA/AMD alone showed none. This GATES the stock
# training experiments (warm-start / broad-universe pretraining): no point training a stock model
# if the broad audit shows no stock signal anywhere.
#
#   AUC ≈ 0.50 → no edge   |   > 0.55 → real signal   |   +cand > base → the new features add edge
#
# The ~16 new stocks DOWNLOAD once (no --skip-download), then cache; crypto + our 8 use cache.
EXTRA_STOCKS = "AVGO QCOM INTC TXN MRVL AMAT LRCX KLAC ADI ON NXPI MCHP WDC QRVO SWKS ASML"
CRYPTO_AUDIT = "BTCUSDT ETHUSDT SOLUSDT"                     # 3 anchors (crypto already characterised)
STOCK_AUDIT  = "SNDK AMD MU AXTI BE NVDA TSM SMCI " + EXTRA_STOCKS
!PRETRAIN_EXTRA_STOCKS="{EXTRA_STOCKS}" python scripts/signal_audit.py --start-year {START_YEAR} --symbols "{CRYPTO_AUDIT} {STOCK_AUDIT}" --horizons "12 48 288"
print('\nIf the broad stock set is ~0.50 across the board → stocks have no 5m technical edge → the')
print('stock specialist should lean on EVENTS (earnings expert) not 5m technicals. If some semis show')
print('>0.55 → broad-universe pretraining is worth it, and we build the asset-class/warm-start infra.')

In [ ]:
# ===== CRYPTO SPECIALIST — clean (HTF-fixed) retrain, crypto-only, backtest at 4h (the edge zone) =====
# Audit verdict: a small real momentum edge at 1h-4h on crypto (~0.53-0.54 AUC); stocks have no
# intraday 5m edge. So we build the CRYPTO specialist first. EVERY prior retrain used the LEAKY HTF
# features — this is the first HONEST crypto baseline. Backtest at H+48 (4h), where the edge is
# biggest vs fixed costs. Judge by ALPHA (excess Sharpe vs buy-and-hold), NOT raw Sharpe.
import shutil
CRYPTO = "BTCUSDT ETHUSDT SOLUSDT AAVEUSDT XLMUSDT XRPUSDT ADAUSDT DOGEUSDT RENDERUSDT NEARUSDT"
!PRETRAIN_NEWS_ALIGN=0 PRETRAIN_EARNINGS_ALIGN=0 NN_TRUNK=lstm python scripts/pretrain.py --mmap --mmap-dir /content/_mmap_cache --amp --skip-download --start-year {START_YEAR} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --seed {SEED} --symbols {CRYPTO}
shutil.copy('models/trading_lstm_latest.pt', 'models/ckpt_crypto.pt'); print('✅ kept models/ckpt_crypto.pt')
!NN_TRUNK=lstm python scripts/backtest.py --checkpoint models/ckpt_crypto.pt --start-year {START_YEAR} --skip-download --primary-horizon 2 --fee-bps 10 --slippage-bps 5 --min-confidence 0.45 --min-edge 0.05 --symbols {CRYPTO} --out-json models/bt_crypto.json
print('\nRead the FINAL VALIDATION METRICS (H+48 expectancy/win-rate) + the backtest ALPHA line.')
print('This is the honest, leak-free crypto baseline. If alpha is ~0/slightly+, meta-labeling is next')
print('(filters to the high-confidence subset to push a thin edge into positive territory).')

In [ ]:
# ===== FINAL COMPREHENSIVE PROBE — best model per asset, per horizon, per conviction =====
# One shot, no NN training. For every (horizon 4h & 1d) x (model gbm/mlp/logistic) x (CRYPTO vs
# STOCKS) it sweeps conviction up to 0.999 and reports ALPHA (excess Sharpe vs B&H, after costs) +
# avg trades. Each model trains ONCE per symbol; thresholds are then free. ~15-25 min.
#
# READ IT AS: where does alpha go clearly POSITIVE at a NON-trivial trade count? That
# (asset, model, horizon, conf) is what we build. Crypto's best model may differ from stocks'.
# 'flat' = the threshold took zero trades (no strategy) — ignore those rows.
!python scripts/gbm_baseline.py --start-year {START_YEAR} --skip-download --models "gbm mlp logistic" --horizons "48 288"

In [ ]:
# ===== CRYPTO LINEAR MODEL — the build the probe pointed to (walk-forward validated) =====
# The crypto edge is LINEAR + thin: a logistic model beat GBM/MLP/LSTM (more capacity = more
# overfit), tradeable only at HIGH conviction. This trains ONE pooled logistic over all 10 crypto
# and validates it WALK-FORWARD across 4 expanding time folds — the regime test the single-split
# probe skipped. Saves the deployable model to models/crypto_linear.npz. ~3-5 min, no GPU.
#
#   Positive alpha in MOST folds → robust, deployable edge → we wire it into the live agent next.
#   Positive only in the last fold → it was regime luck → we adjust horizon/conf or rethink.
!python scripts/train_linear.py --start-year {START_YEAR} --skip-download --horizon 12 --conf 0.65 --folds 4
print('\nAlso try a couple of variants to find the most ROBUST (positive in the most folds), e.g.:')
print('  !python scripts/train_linear.py --skip-download --horizon 1  --conf 0.70 --folds 5')
print('  !python scripts/train_linear.py --skip-download --horizon 48 --conf 0.65 --folds 4')

In [ ]:
# ===== MARKET-NEUTRAL CROSS-SECTIONAL — the regime-agnostic shot =====
# Every DIRECTIONAL model died on regime shift (walk-forward exposed the linear model as luck). A
# DOLLAR-NEUTRAL long-short book trades RELATIVE strength (long the strongest k alts, short the
# weakest k, equal dollars) instead of market direction — structurally regime-stable. No training,
# no new data. Reports portfolio Sharpe (market-neutral → Sharpe IS the edge, no beta to subtract)
# across walk-forward folds, sweeping momentum vs reversal x lookback/hold. ~2 min.
#
#   POSITIVE Sharpe in MOST folds (4-5/5) for some config → a real, regime-robust edge → we build it
#       into the crypto specialist (rank-and-trade) + portfolio manager.
#   Every config flips sign across folds → crypto has no robust cross-sectional edge either →
#       we pivot to events (earnings expert) / new data (funding, on-chain).
!python scripts/cross_sectional.py --start-year {START_YEAR} --skip-download --k 3 --folds 5

In [ ]:
# ===== Experiment 1 — LSTM baseline (news OFF, earnings OFF, trunk=lstm) =====
import shutil
!PRETRAIN_NEWS_ALIGN=0 PRETRAIN_EARNINGS_ALIGN=0 NN_TRUNK=lstm python scripts/pretrain.py --mmap --mmap-dir /content/_mmap_cache --amp --skip-download --start-year {START_YEAR} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --seed {SEED} --symbols {SYMBOLS}
shutil.copy('models/trading_lstm_latest.pt', 'models/ckpt_lstm_baseline.pt'); print('✅ kept models/ckpt_lstm_baseline.pt')
!PRETRAIN_NEWS_ALIGN=0 PRETRAIN_EARNINGS_ALIGN=0 NN_TRUNK=lstm python scripts/backtest.py --checkpoint models/ckpt_lstm_baseline.pt --start-year {START_YEAR} --skip-download --primary-horizon {PRIMARY_H} --fee-bps 10 --slippage-bps 5 --min-confidence 0.45 --min-edge 0.05 --out-json models/bt_lstm_baseline.json

In [ ]:
# ===== Experiment 2 — LSTM + semantic news embeddings (news ON, trunk=lstm) =====
# Fetches + embeds historical news per symbol (mainly helps the stocks). The build is slower.
import shutil
!PRETRAIN_NEWS_ALIGN=1 PRETRAIN_EARNINGS_ALIGN=0 NN_TRUNK=lstm python scripts/pretrain.py --mmap --mmap-dir /content/_mmap_cache --amp --skip-download --start-year {START_YEAR} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --seed {SEED} --symbols {SYMBOLS}
shutil.copy('models/trading_lstm_latest.pt', 'models/ckpt_lstm_news.pt'); print('✅ kept models/ckpt_lstm_news.pt')
# Backtest MUST also set NEWS_ALIGN=1 so the news columns are populated (else it feeds zeros the model never saw).
!PRETRAIN_NEWS_ALIGN=1 PRETRAIN_EARNINGS_ALIGN=0 NN_TRUNK=lstm python scripts/backtest.py --checkpoint models/ckpt_lstm_news.pt --start-year {START_YEAR} --skip-download --primary-horizon {PRIMARY_H} --fee-bps 10 --slippage-bps 5 --min-confidence 0.45 --min-edge 0.05 --out-json models/bt_lstm_news.json

In [ ]:
# ===== Experiment 3 — TCN trunk (news OFF, trunk=tcn) — architecture A/B vs Experiment 1 =====
import shutil
!PRETRAIN_NEWS_ALIGN=0 PRETRAIN_EARNINGS_ALIGN=0 NN_TRUNK=tcn python scripts/pretrain.py --mmap --mmap-dir /content/_mmap_cache --amp --skip-download --start-year {START_YEAR} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --seed {SEED} --symbols {SYMBOLS}
shutil.copy('models/trading_lstm_latest.pt', 'models/ckpt_tcn.pt'); print('✅ kept models/ckpt_tcn.pt')
!PRETRAIN_NEWS_ALIGN=0 PRETRAIN_EARNINGS_ALIGN=0 NN_TRUNK=tcn python scripts/backtest.py --checkpoint models/ckpt_tcn.pt --start-year {START_YEAR} --skip-download --primary-horizon {PRIMARY_H} --fee-bps 10 --slippage-bps 5 --min-confidence 0.45 --min-edge 0.05 --out-json models/bt_tcn.json

In [ ]:
# ===== Overview — compare the three experiments side by side =====
import json, os
rows = [('lstm_baseline', 'LSTM (news off)'), ('lstm_news', 'LSTM + news'), ('tcn', 'TCN (news off)')]
print(f"{'experiment':20s} {'horizon':8s} {'ALPHA(medEx)':>12s} {'medSharpe':>10s} {'medB&H':>8s} {'syms':>5s}")
print('-' * 72)
for key, label in rows:
    p = f'models/bt_{key}.json'
    if not os.path.exists(p):
        print(f'{label:20s} (not run yet — run its cell above)'); continue
    d = json.load(open(p)); a = d['aggregate']
    al = 'n/a' if a.get('median_excess_sharpe') is None else f"{a['median_excess_sharpe']:+.2f}"
    md = 'n/a' if a.get('median_sharpe')        is None else f"{a['median_sharpe']:.2f}"
    bh = 'n/a' if a.get('mean_bench_sharpe')    is None else f"{a['mean_bench_sharpe']:.2f}"
    print(f"{label:20s} {d['horizon_label']:8s} {al:>12s} {md:>10s} {bh:>8s} {a['symbols']:>5d}")
print('\nALPHA (median excess Sharpe vs buy-and-hold) is the verdict — the ONLY skill measure here.')
print('  alpha > 0 across the board   = real predictive edge (keep that checkpoint).')
print('  alpha ≈ 0 while medSharpe≈medB&H = the model is just riding market beta in a rising test')
print('  window, NOT predicting — the giant Sharpes/returns are the market, not skill. Compare ALPHA.')

In [ ]:
# ===== Which checkpoints to keep in Drive =====
import os, glob, shutil
print('Checkpoints currently in Drive models/:')
for f in sorted(glob.glob('models/*.pt')):
    print(f'  {f:44s} {os.path.getsize(f)/1e6:6.1f} MB')

print('''
KEEP these three named archives (your experiment results, safe on Drive):
  models/ckpt_lstm_baseline.pt   LSTM, news off
  models/ckpt_lstm_news.pt       LSTM, news on
  models/ckpt_tcn.pt             TCN,  news off

The live agent ALWAYS loads  models/trading_lstm_latest.pt  → promote your WINNER to that name.
Intermediates you can delete:  models/pretrain_v2_best*.pt
''')

# 1) Promote the winner for live use. EDIT `WINNER` to the best row from the overview, then run:
WINNER = 'models/ckpt_lstm_news.pt'            # <-- set to your best-Sharpe checkpoint
shutil.copy(WINNER, 'models/trading_lstm_latest.pt')
print('✅ promoted', WINNER, '-> models/trading_lstm_latest.pt  (this is what the live agent loads)')

# 2) Optional cleanup of intermediates (uncomment to run):
# for f in glob.glob('models/pretrain_v2_best*.pt'): os.remove(f); print('removed', f)